# wbi-birds — packaged example

End-to-end demo on the packaged fixtures (no private data):
**ingest → fit → compile → build → scenario → parity**.

Run from a repository checkout with the engine installed:

    python -m venv .venv && . .venv/bin/activate
    pip install -e ".[dev,workflow,examples]"

The example stages an eBird-derived table and provided coefficients into a
fixture instance (TSA24), builds the ws3 model, runs one harvest scenario with
the landscape-scope bird row, and verifies the result against a packaged
reference table.


## 1. Stage a work directory from the packaged example data

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "workflows" / "wbi_example.yaml").is_file():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Run this notebook from the repository checkout.")
    REPO_ROOT = REPO_ROOT.parent
print(f"repository: {REPO_ROOT}")

WORKDIR = REPO_ROOT / "examples" / "_workdir"
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
for rel in ("instances", "bundles", "data", "assets", "references"):
    (WORKDIR / rel).mkdir(parents=True)

shutil.copytree(REPO_ROOT / "examples/data/instance_template", WORKDIR / "instances/tsa24")
shutil.copytree(REPO_ROOT / "examples/data/femic_bundle", WORKDIR / "bundles/tsa24")
shutil.copyfile(REPO_ROOT / "examples/data/ebd/sample_ebd.txt", WORKDIR / "data/sample_ebd.txt")
shutil.copyfile(
    REPO_ROOT / "examples/data/instance_template/bird/coefficients.yaml",
    WORKDIR / "assets/coefficients.yaml",
)
shutil.copyfile(REPO_ROOT / "examples/data/reference/tsa24-s00.csv", WORKDIR / "references/suite.csv")

print(f"workdir: {WORKDIR}")

## 2. Run the chain (FreshForge)

The graph `workflows/wbi_example.yaml` chains six provider nodes; the run record
and evidence manifest capture every input, output, artifact, and diagnostic.

In [ ]:
run = subprocess.run(
    [
        sys.executable,
        "-m",
        "freshforge",
        "run",
        "workflows/wbi_example.yaml",
        "--workdir",
        str(WORKDIR),
        "--json",
        "--evidence",
        str(WORKDIR / "evidence.json"),
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    # strip colour-forcing vars so the JSON stays machine-readable
    # (Rich forces colour whenever FORCE_COLOR / CLICOLOR_FORCE is present)
    env={
        key: value
        for key, value in os.environ.items()
        if key not in {"FORCE_COLOR", "CLICOLOR_FORCE"}
    },
)
if run.returncode != 0:
    raise RuntimeError(run.stdout[-2000:] + "\n" + run.stderr[-2000:])
record = json.loads(run.stdout)
assert record["ok"] is True
assert (WORKDIR / "evidence.json").is_file()
for node in record["run"]["nodes"]:
    print(f"{node['id']:<18} {node['status']}")

## 3. Scenario results

Per-period harvest volume (m³) and landscape-scope bird abundance for scenario
`tsa24-s00`.

In [ ]:
import pandas as pd

table = pd.read_csv(WORKDIR / "instances/tsa24/results/scenarios/tsa24/tsa24-s00.csv")
table.head()

## 4. Parity check

`verify_parity` compares the scenario tables against `references/suite.csv`
(the packaged reference generated from this fixture chain) under the tolerance
policy `|engine - reference| <= abs_tol + rel_tol * |reference|`. In the
campaign, the reference comes from the legacy RBNU/5-TSA archive.

In [ ]:
parity_node = next(node for node in record["run"]["nodes"] if node["id"] == "verify_parity")
print(json.dumps(parity_node["outputs"]["parity_report"], indent=2))
print((WORKDIR / "instances/tsa24/results/parity/tsa24-demo-parity.md").read_text())

## 5. Harvest and bird series

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(table["period"], table["harvest_volume_m3"], marker="o", label="harvest volume")
ax.set_xlabel("period")
ax.set_ylabel("harvest volume (m³)", color="tab:blue")
ax.tick_params(axis="y", labelcolor="tab:blue")

birds = ax.twinx()
birds.plot(table["period"], table["bird_abundance"], marker="s", color="tab:orange")
birds.set_ylabel("bird abundance (landscape)", color="tab:orange")
birds.tick_params(axis="y", labelcolor="tab:orange")

ax.set_title("Scenario tsa24-s00 — harvest vs landscape bird abundance")
fig.tight_layout()

## Next steps

- The campaign runs one instance per TSA (`workflows/wbi_mu_matrix.yaml`).
- Real parity with the legacy archive:
  `planning/parity-reference-acquisition.md`.
- Docs site: quickstart, concepts, workflows, parity, API reference.
